In [1]:
import sys
print(sys.executable)
!{sys.executable} -m pip install plotly -q
import plotly
print("plotly", plotly.__version__)

/Users/nivbenavraham/.local/share/virtualenvs/beehero-model-monitoring-zhiPBxK4/bin/python

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
plotly 6.6.0


In [2]:
import pandas as pd

In [3]:
from pathlib import Path

RESULTS_DIR = Path("/Users/nivbenavraham/dev/model-monitor-analysis/data/results")

def load_all_sensors() -> pd.DataFrame:
    """All sensors across all groups and dates (temp_metric_phase1.parquet)."""
    return pd.read_parquet(RESULTS_DIR / "temp_metric_phase1.parquet")

def load_group_date_sensors(group_id: int, date: str) -> pd.DataFrame:
    """
    Sensor-level results for a specific group and date.
    date format: 'YYYY-MM-DD'
    """
    path = RESULTS_DIR / f"group_{group_id}" / date / f"{group_id}_{date}_temp_metric_summary.parquet"
    if not path.exists():
        raise FileNotFoundError(f"No summary found: {path}")
    return pd.read_parquet(path)

def load_group_summary(group_id: int) -> pd.DataFrame:
    """All per-date summaries for a group, concatenated."""
    parts = sorted((RESULTS_DIR / f"group_{group_id}").rglob(f"{group_id}_*_temp_metric_summary.parquet"))
    if not parts:
        raise FileNotFoundError(f"No summary files found for group {group_id}")
    return pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)

def load_all_summaries() -> pd.DataFrame:
    """All per-(group, date) summaries concatenated into one DataFrame."""
    parts = sorted(RESULTS_DIR.rglob("*_temp_metric_summary.parquet"))
    if not parts:
        raise FileNotFoundError("No summary files found in data/results/")
    return pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)


In [4]:
# --- Examples ---

# All sensor rows (17k+ rows)
all_sensors = load_all_sensors()
print(f"all_sensors: {all_sensors.shape}")

# Summary for one group + date  (one row per hive_size_bucket)
summary_491_16 = load_group_date_sensors(491, "2026-02-16")
print(f"\ngroup 491 / 2026-02-16 summary:")
display(summary_491_16)

# All dates for one group
group_491 = load_group_summary(491)
print(f"\ngroup 491 all dates: {group_491.shape}")
display(group_491)

# Everything in one table
all_summaries = load_all_summaries()
print(f"\nall summaries: {all_summaries.shape}")
display(all_summaries.head(10))


all_sensors: (17805, 10)

group 491 / 2026-02-16 summary:


,group_id,date,hive_size,std_dev,ambient_correlation,mean_temp,percent_comfort,PASS,WARNING,FAIL
0,491,2026-02-16,large,0.18,-0.32,34.76,99.99,615,0,1
1,491,2026-02-16,medium,1.76,0.36,31.37,62.50,22,0,0
2,491,2026-02-16,small,3.88,0.85,15.37,0.00,13,2,0



group 491 all dates: (12, 10)


,group_id,date,hive_size,std_dev,ambient_correlation,mean_temp,percent_comfort,PASS,WARNING,FAIL
0,491,2026-02-16,large,0.18,-0.32,34.76,99.99,615,0,1
1,491,2026-02-16,medium,1.76,0.36,31.37,62.50,22,0,0
2,491,2026-02-16,small,3.88,0.85,15.37,0.00,13,2,0
3,491,2026-02-17,large,0.19,0.08,34.70,99.92,616,0,4
4,491,2026-02-17,medium,1.79,0.51,30.71,50.44,19,0,0
5,491,2026-02-17,small,2.46,0.84,13.77,0.00,12,3,0
6,491,2026-02-18,large,0.22,0.47,34.57,99.81,426,0,156
7,491,2026-02-18,medium,1.18,0.71,31.68,67.93,48,0,0
8,491,2026-02-18,small,3.02,0.87,13.15,1.69,13,3,0
9,491,2026-02-19,large,0.14,-0.44,34.44,99.96,555,0,1



all summaries: (42, 10)


,group_id,date,hive_size,std_dev,ambient_correlation,mean_temp,percent_comfort,PASS,WARNING,FAIL
0,491,2026-02-16,large,0.18,-0.32,34.76,99.99,615,0,1
1,491,2026-02-16,medium,1.76,0.36,31.37,62.50,22,0,0
2,491,2026-02-16,small,3.88,0.85,15.37,0.00,13,2,0
3,491,2026-02-17,large,0.19,0.08,34.70,99.92,616,0,4
4,491,2026-02-17,medium,1.79,0.51,30.71,50.44,19,0,0
5,491,2026-02-17,small,2.46,0.84,13.77,0.00,12,3,0
6,491,2026-02-18,large,0.22,0.47,34.57,99.81,426,0,156
7,491,2026-02-18,medium,1.18,0.71,31.68,67.93,48,0,0
8,491,2026-02-18,small,3.02,0.87,13.15,1.69,13,3,0
9,491,2026-02-19,large,0.14,-0.44,34.44,99.96,555,0,1


In [5]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"
from plotly.subplots import make_subplots

GROUP_ID = 491
DATE     = "2026-02-16"
BASE     = Path(f"/Users/nivbenavraham/dev/model-monitor-analysis/data/samples/group_{GROUP_ID}/{DATE}")

sensor_df  = pd.read_parquet(BASE / f"{GROUP_ID}_{DATE}_sensor_temperature.parquet")
gateway_df = pd.read_parquet(BASE / f"{GROUP_ID}_{DATE}_gateway_temperature.parquet")

# ── downsample to keep the plot responsive (every 10th point) ────────────────
def sample_df(df, n=10):
    return df.iloc[::n].copy() if len(df) > n else df

gateway_plot = sample_df(gateway_df)
sensor_plot  = sample_df(sensor_df)

small_sensors  = sensor_df[sensor_df["hive_size_bucket"] == "small" ]["sensor_mac_address"].unique()
medium_sensors = sensor_df[sensor_df["hive_size_bucket"] == "medium"]["sensor_mac_address"].unique()
large_sensors  = sensor_df[sensor_df["hive_size_bucket"] == "large" ]["sensor_mac_address"].unique()

fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=[
        f"GW Temps — {gateway_df['gateway_mac_address'].nunique()} gateways",
        f"Small hives (<6 frames) — {len(small_sensors)} sensors",
        f"Medium hives (6-10 frames) — {len(medium_sensors)} sensors",
        f"Large hives (10+ frames) — {len(large_sensors)} sensors",
    ],
    shared_yaxes=True,
    horizontal_spacing=0.06,
)

# ── Col 1: Gateway ambient ────────────────────────────────────────────────────
if not gateway_plot.empty:
    hw_ext = gateway_plot.get("hardware_extension_present", pd.Series([False] * len(gateway_plot)))
    colors = ["red" if v else "blue" for v in hw_ext]
    fig.add_trace(go.Scatter(
        x=gateway_plot["timestamp"],
        y=gateway_plot["pcb_temperature_two"],
        mode="markers", marker=dict(size=3, color=colors),
        name="Gateways", showlegend=False,
        text=gateway_plot["gateway_mac_address"],
        hovertemplate="<b>%{text}</b><br>Temp: %{y:.1f}°C<extra></extra>",
    ), row=1, col=1)

# ── Col 2: Small hives ────────────────────────────────────────────────────────
small_data = sensor_plot[sensor_plot["sensor_mac_address"].isin(small_sensors)]
if not small_data.empty:
    fig.add_trace(go.Scatter(
        x=small_data["timestamp"],
        y=small_data["pcb_temperature_one"],
        mode="markers", marker=dict(size=3, color="#e74c3c"),
        name="Small", showlegend=False,
        text=small_data["sensor_mac_address"],
        hovertemplate="<b>%{text}</b><br>Temp: %{y:.1f}°C<extra></extra>",
    ), row=1, col=2)
    fig.add_hline(y=26, line_dash="dash", line_color="green", line_width=2,
                  annotation_text="26°C", annotation_position="top right",
                  row=1, col=2)

# ── Col 3: Medium hives ───────────────────────────────────────────────────────
medium_data = sensor_plot[sensor_plot["sensor_mac_address"].isin(medium_sensors)]
if not medium_data.empty:
    fig.add_trace(go.Scatter(
        x=medium_data["timestamp"],
        y=medium_data["pcb_temperature_one"],
        mode="markers", marker=dict(size=3, color="#f39c12"),
        name="Medium", showlegend=False,
        text=medium_data["sensor_mac_address"],
        hovertemplate="<b>%{text}</b><br>Temp: %{y:.1f}°C<extra></extra>",
    ), row=1, col=3)
    fig.add_hline(y=26, line_dash="dash", line_color="green", line_width=2,
                  annotation_text="26°C", annotation_position="top right",
                  row=1, col=3)
    fig.add_hline(y=32, line_dash="dash", line_color="green", line_width=2,
                  annotation_text="32°C", annotation_position="top right",
                  row=1, col=3)

# ── Col 4: Large hives ────────────────────────────────────────────────────────
large_data = sensor_plot[sensor_plot["sensor_mac_address"].isin(large_sensors)]
if not large_data.empty:
    fig.add_trace(go.Scatter(
        x=large_data["timestamp"],
        y=large_data["pcb_temperature_one"],
        mode="markers", marker=dict(size=3, color="#2ecc71"),
        name="Large", showlegend=False,
        text=large_data["sensor_mac_address"],
        hovertemplate="<b>%{text}</b><br>Temp: %{y:.1f}°C<extra></extra>",
    ), row=1, col=4)
    fig.add_hline(y=28, line_dash="dash", line_color="green", line_width=2,
                  annotation_text="28°C", annotation_position="top right",
                  row=1, col=4)
    fig.add_hline(y=35, line_dash="dash", line_color="green", line_width=2,
                  annotation_text="35°C", annotation_position="top right",
                  row=1, col=4)

fig.update_xaxes(title_text="Time")
fig.update_yaxes(title_text="Temperature (°C)", col=1)

fig.update_layout(
    title=f"Group {GROUP_ID} | {DATE} — Sensor Temperature Patterns by Hive Size",
    showlegend=False,
    height=500,
    template="plotly_white",
)

fig.show()


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed